# 기능상속 AI 상담 — 질문 이해 모델 학습 (Colab)

한국어 인코더(`klue/roberta-small`)를 파인튜닝해 상담 질문에서 **의도 · 가게/마을/물건 · 물건 분류**를 한 번에 뽑는 모델을 만듭니다.

**시작 전:** 메뉴 `런타임 → 런타임 유형 변경 → T4 GPU`를 고르세요. 위에서부터 차례로 실행(Shift+Enter)하면 됩니다. 전체 20~30분.

| 시험지 | 뜻 |
|---|---|
| `test_seen` | 학습과 같은 문장 틀·어휘 (쉬움, 참고용) |
| `test_unseen` | **처음 보는 문장 틀 + 처음 보는 물건·가게·마을** (실력 확인용) |
| `human_test` | 사람이 쓴 문장 (최종 시험지 — `model/data/human_test.csv`에 팀원이 계속 추가) |

In [ ]:
!nvidia-smi -L

## 1. 코드·데이터 받기

In [ ]:
!git clone -q https://github.com/chojeongbin0308-ops/gineung.git
%cd gineung
!pip -q install onnx onnxruntime

## 2. 데이터 만들기와 규칙 기준선 채점
지금 사이트의 키워드 규칙 방식이 같은 시험지에서 몇 점인지 먼저 잽니다.

In [ ]:
!python model/make_dataset.py
!python model/human_to_jsonl.py
!python model/rule_baseline.py

## 3. 학습 · 평가 · ONNX 내보내기
`--model klue/roberta-base`로 바꾸면 더 크고 정확한 모델(학습 시간 약 2배)을 비교해 볼 수 있습니다.

In [ ]:
!python model/train_intent.py --model klue/roberta-small --epochs 5 --out runs/roberta-small

## 4. 결과 표 (모델 vs 규칙)

In [ ]:
import json, pandas as pd
m = json.load(open('runs/roberta-small/metrics.json', encoding='utf-8'))
rows = []
for k, v in m['test'].items():
    b = m.get('rule_baseline', {}).get(k, {})
    rows.append({'시험지': k, '문장 수': v['n'], '의도 정확도(모델)': v['intent_acc'], '의도 정확도(규칙)': b.get('intent_acc'),
                 '물건분류(모델)': v['item_cat_acc'], '물건분류(규칙)': b.get('item_cat_acc'), '개체 F1(모델)': v['entity_f1']})
display(pd.DataFrame(rows))
print('ONNX 크기:', m.get('export'))

## 5. 직접 물어보기

In [ ]:
import sys, torch
sys.path.insert(0, 'model')
import train_intent as T
from transformers import AutoTokenizer
dev = 'cuda' if torch.cuda.is_available() else 'cpu'
tok = AutoTokenizer.from_pretrained('runs/roberta-small')
mdl = T.Joint('klue/roberta-small').to(dev)
mdl.load_state_dict(torch.load('runs/roberta-small/best.pt', map_location=dev))
predict = T.predict_fn(mdl, tok, dev)
for q in ['하리농약사 닫으믄 고추약은 어디서 사능가', '근처에 모짜렐라 파는 데 있소', '오는 길에 계란 좀 사다 주쇼',
          '구룡리는 뭐가 제일 구멍이여', '농약 팔라믄 무슨 허가가 있어야 되나']:
    print(q, '→', predict(q))

## 6. 결과 내려받기
`intent_model.zip`을 받아 GitHub 저장소의 `model/release/`에 풀어 넣으면 사이트에 연결할 수 있습니다. (용량이 커서 업로드는 Claude에게 맡겨도 됩니다.)

In [ ]:
!cd runs/roberta-small && zip -qr ../../intent_model.zip onnx/model_quantized.onnx tokenizer.json tokenizer_config.json labels.json metrics.json
!ls -la intent_model.zip
from google.colab import files
files.download('intent_model.zip')

### (선택) Hugging Face에 공개
토큰은 이 노트북에만 입력하고 어디에도 저장·공유하지 마세요.

In [ ]:
# from huggingface_hub import login, upload_folder
# login()   # 본인 토큰 입력
# upload_folder(folder_path='runs/roberta-small', repo_id='본인아이디/gineung-intent-ko', allow_patterns=['onnx/*','tokenizer*','labels.json','metrics.json'])